# Setup

In [0]:
%pip install nhp_data

In [0]:
from pyspark.sql import functions as F
from nhp.data.get_spark import get_spark
from nhp.data.table_names import table_names
from nhp.data.inputs_data.ip.rates import get_population
from nhp.data.inputs_data.direct_standardisation import directly_standardise
from nhp.data.inputs_data.helpers import complete_age_sex_rows
from nhp.data.raw_data.mitigators.ip.hes_datasets import (
    any_diagnosis,
    nhp_apc,
    primary_diagnosis
)
from pyspark.sql.window import Window

In [0]:
# Lakemart where we're saving outputs
lakeNameSU = "udalstdataanalysisprod.dfs.core.windows.net"
containerNameSU = "analytics-projects"
tpma_review_path = "abfss://"+containerNameSU+"@"+lakeNameSU+"/StrategyUnit/StrategyUnit/analytics_modelling/nhp_tpma_review/2026_09_discharge_ready_date/"

# Getting data

For each qualifying admission*, calculating the duration of delay as the difference between the modified discharge ready date^ and the discharge date. Duration of delays capped at 120 days.

*Qualifying admission:
- the patient was under 16 years old.
- the length of stay was 0 days.
- the method of admission was other than elective or emergency.
- the patient died in hospital or was transferred to other hospitals/hospices.
- the treatment function groups was not “specific acute”.

^Modified discharge ready date:
- NULL discharge ready dates replaced with the discharge date.
- Discharge ready dates that happen after the discharge date replaced with the discharge date.
- Discharge ready dates of 1900-01-01 are replaced with the discharge date.



In [0]:
def get_discharge_delay():
    spark = get_spark()

    data = (
        spark.read.table(table_names.hes_apc)
        .withColumn("Der_Activity_Month", F.regexp_replace(F.substring(F.col("disdate"), 1, 7), "-", ""))
        .filter(F.col("last_episode_in_spell") & ~F.col("disdate").isNull() & (F.col("Der_Activity_Month") >= 202309))
        # exclude under 16s, applying the age logic from https://github.com/The-Strategy-Unit/nhp_data/blob/main/src/nhp/data/nhp_datasets/apc.py:
        .withColumn(
            "age",
            F.when(
                (F.col("admiage") == 999) | F.col("admiage").isNull(),
                F.when(F.col("startage") > 7000, 0).otherwise(F.col("startage")),
            ).otherwise(F.col("admiage")),
        )
        .filter(F.col("age") >= 16)
        # exclude zero length of stay:
        .filter(F.datediff(F.col("disdate"), F.col("admidate")) > 0)
        # exclude method of admission other than elective or emergency:
        .filter(~F.col("admimeth").isin("31", "32", "82", "83"))
        # exclude those who die in hospital and those transferred to other hospitals/hospices:
        .filter(~F.col("disdest").isin("51", "52", "87", "88", "79"))
        .filter(~F.col("dismeth").isin(4, 5))    
        # filter to "specific acute" treatment function groups (see appendix for more info):
        .filter(F.col("tretspef").isin(
            '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '115', '120', '130', '140', '141', '142', '143', '144', '145', '150', '160', '161', '170', '171', '172', '173', '174', '180', '190', '191', '192', '200', '211', '212', '213', '214', '215', '216', '217', '218', '219', '220', '221', '222', '230', '240', '241', '242', '250', '251', '252', '253', '254', '255', '256', '257', '258', '259', '261', '262', '263', '264', '270', '280', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '313', '314', '315', '316', '317', '318', '319', '320', '321', '322', '323', '324', '325', '326', '327', '328', '329', '330', '333', '340', '341', '342', '343', '347', '348', '350', '352', '361', '370', '371', '400', '401', '410', '420', '421', '422', '430', '431', '450', '451', '460', '461', '502', '503', '505', '663', '670', '673', '675', '677', '800', '811', '812', '822', '834'
            ))
        # replace NULL discharge ready dates with discharge dates and replace discharge ready dates after the discharge dates with discharge dates:
        .withColumn(
            "disreadydate_valid", 
            F.when(F.col("disreadydate").isNull(), F.col("disdate"))
            .when(F.col("disreadydate") > F.col("disdate"), F.col("disdate"))
            .when(F.col("disreadydate") == "1900-01-01", F.col("disdate"))
            .otherwise(F.col("disreadydate"))
            )
        # Calculate delay duration:
        .withColumn("delay_duration", F.datediff(F.col("disdate"), F.col("disreadydate_valid")))
        # Cap delay durations:
        .withColumn(
            "delay_duration_capped",
            F.when(F.col("delay_duration") > 120, 120)
            .otherwise(F.col("delay_duration"))
            )
    )

    return(data)

#get_discharge_delay().limit(10).display()

# Getting age and sex data

Now aggregating to get counts of admissions by age_band, sex and delay duration.

In [0]:
delayed_discharge_age_sex = (
    get_discharge_delay()
    .withColumn(
            "age_band",
            F.when(F.col("age").between(16, 19), "16-19")
            .when(F.col("age").between(20, 24), "20-24")
            .when(F.col("age").between(25, 29), "25-29")
            .when(F.col("age").between(30, 34), "30-34")
            .when(F.col("age").between(35, 39), "35-39")
            .when(F.col("age").between(40, 44), "40-44")
            .when(F.col("age").between(45, 49), "45-49")
            .when(F.col("age").between(50, 54), "50-54")
            .when(F.col("age").between(55, 59), "55-59")
            .when(F.col("age").between(60, 64), "60-64")
            .when(F.col("age").between(65, 69), "65-69")
            .when(F.col("age").between(70, 74), "70-74")
            .when(F.col("age").between(75, 79), "75-79")
            .when(F.col("age").between(80, 84), "80-84")
            .when(F.col("age").between(85, 89), "85-89")
            .when(F.col("age").between(90, 120), "90+")
            .otherwise(F.lit(None))
        )
    .withColumn(
        "sex",
        F.when(F.col("sex").isin("1", "2"), F.col("sex"))
        .otherwise(F.lit(None))
    )
    .groupBy("Der_Activity_Month", "delay_duration_capped", "age_band", "sex")
    .agg(F.count("Der_Activity_Month").alias("n"))
)

In [0]:
# Write parquet file
delayed_discharge_age_sex.write.mode("overwrite").parquet(tpma_review_path + "delayed_discharge_age_sex.parquet")

# By admission method

In [0]:
non_elective = ((F.col("admimeth").rlike("^2")) & (F.col("classpat") == 1))
elective = ((F.col("admimeth").rlike("^1")) & (F.col("classpat") == 1))

In [0]:
delayed_discharge_pod = (
    get_discharge_delay()
    .withColumn(
        "pod",
        F.when(non_elective, "non_elective")
        .when(elective, "elective")
        .otherwise("other")
    )
    .filter(F.col("pod").isin("elective", "non_elective"))
    .groupBy("Der_Activity_Month", "pod")
    .agg(
        F.sum("delay_duration_capped").alias("total"),
        F.count("Der_Activity_Month").alias("n")
        )
)

# Write parquet file
delayed_discharge_pod.write.mode("overwrite").parquet(tpma_review_path + "delayed_discharge_pod.parquet")

# By specialty group

Using the treatment function codes for "Surgical specialties" and "Medical specialties" in https://archive.datadictionary.nhs.uk/DD%20Release%20March%202022/supporting_information/main_specialty_and_treatment_function_codes_table.html. All other codes are "Other".

Note: some of the treatment function codes here have been excluded by the filter to "specific acute" treatment function groups earlier. So data for these treatment function codes will not be included in outputs.

In [0]:
# some of these codes are not in the data because filter to "specific acute" treatment function groups 
medical = (F.col("tretspef").isin(
    '180', '190', '191', '192', '200', '300', '301', '302', '303', '304', '305', '306', '307', '308', '309', '310', '311', '312', '313', '314', '315', '316', '317', '318', '319', '320', '322', '323', '324', '325', '326', '327', '328', '329', '330', '331', '333', '335', '340', '341', '342', '343', '344', '345', '346', '347', '348', '350', '352', '360', '361', '370', '371', '400', '401', '410', '420', '422', '424', '430', '431', '450', '451', '460', '461', '500', '501', '502', '503', '504', '505', '510', '520', '600', '610', '620', '834'
))

surgical = (F.col("tretspef").isin(
    '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '113', '115', '120', '130', '140', '141', '143', '144', '145', '149', '150', '160', '161', '170', '172', '173', '174'
))

In [0]:
delayed_discharge_specialty_group = (
    get_discharge_delay()
    .withColumn(
        "specialty_group",
        F.when(medical, "medical")
        .when(surgical, "surgical")
        .otherwise("other")
    )
    .groupBy("Der_Activity_Month", "specialty_group")
    .agg(
        F.sum("delay_duration_capped").alias("total"),
        F.count("Der_Activity_Month").alias("n")
        )
)

# Write parquet file
delayed_discharge_specialty_group.write.mode("overwrite").parquet(tpma_review_path + "delayed_discharge_specialty_group.parquet")

# Number with non-zero delay

In [0]:
delayed_discharge_non_zero = (
    get_discharge_delay()
    .filter(F.col("fyear") == "202425")
    .withColumn(
        "non_zero_delay", 
        F.when(F.col("delay_duration_capped") > 0, F.lit(1))
        .otherwise(F.lit(0))
        )
    .withColumn(
        "pod",
        F.when(non_elective, "non_elective")
        .when(elective, "elective")
        .otherwise("other")
    )
    .withColumn(
        "specialty_group",
        F.when(medical, "medical")
        .when(surgical, "surgical")
        .otherwise("other")
    )
    .groupBy("non_zero_delay", "pod", "specialty_group")
    .count()
)

# Write parquet file
delayed_discharge_non_zero.write.mode("overwrite").parquet(tpma_review_path + "delayed_discharge_non_zero.parquet")

# Top n
Getting top n primary diagnoses and procedures for admissions with a delayed discharge in 24/25.

In [0]:
def get_top_n_delayed_discharge(data, type, n):
    top_n = (
        data
        .filter((F.col("fyear") == 202425) & (~F.col(type).isNull()))
        .groupBy(type)
        .agg(F.count("fyear").alias("n"))
        .withColumn("perc", F.col("n") / F.sum("n").over(Window.partitionBy()))
        .sort(F.col("n").desc())
        .limit(n)        
        )

    return(top_n)

In [0]:
# Getting primary procedure:
df_primary_diagnosis = (
    spark.read.table(table_names.hes_apc_diagnoses)
    .filter(F.col("diag_order") == 1)
    .withColumn("primary_diagnosis", F.left(F.col("diagnosis"), F.lit(3)))
    .select("epikey", "fyear", "procode3", "primary_diagnosis")
    )

# Getting primary procedure, adapted from https://github.com/The-Strategy-Unit/nhp_data/blob/cd88d8ce432b877cefdee134d7eee1280e52ac57/src/nhp/data/nhp_datasets/apc.py#L54:
df_primary_procedure = (
    spark.read.table(table_names.hes_apc_procedures)
    .filter(F.col("procedure_order") == 1)
    .filter(~F.col("procedure_code").rlike("^O(1[1-46]|28|3[01346]|4[2-8]|5[23])"))
    .filter(~F.col("procedure_code").rlike("^X[6-9]"))
    .filter(~F.col("procedure_code").rlike("^[UYZ]"))
    .withColumn("primary_procedure", F.left(F.col("procedure_code"), F.lit(3)))
    .select("epikey", "fyear", "procode3", "primary_procedure")
)

data_for_delayed_discharge_top_n = (
    get_discharge_delay()
    .filter(F.col("delay_duration_capped") > 0)
    .join(df_primary_diagnosis, ["epikey", "fyear", "procode3"], "left")
    .join(df_primary_procedure, ["epikey", "fyear", "procode3"], "left")
)

delayed_discharge_top_n = (
    get_top_n_delayed_discharge(
        data_for_delayed_discharge_top_n, 
        "primary_diagnosis", 
        6
        )
    .withColumn("type", F.lit("diagnosis"))
    .union(
        get_top_n_delayed_discharge(
            data_for_delayed_discharge_top_n, 
            "primary_procedure", 
            6
            )
        .withColumn("type", F.lit("procedure"))
    )
    .withColumnRenamed("primary_diagnosis", "code")
)

# Write parquet file
delayed_discharge_top_n.write.mode("overwrite").parquet(tpma_review_path + "delayed_discharge_top_n.parquet")  